### Testing RAG Applications - (Advanced ⚡️) 📑

#### RAG Application
This application reads data about Model Context Protocol (MCP) server from internet, stores in vector stores, chunks the data with embedding and useful to answer the question about MCP while inferenced.

<img src="./img/RAG.png" width="500" height="400" style="display: block; margin: auto;">

In [1]:
# Base packages for this section
!pip install -qU langchain-core langchain-classic langchain-text-splitters langchain-community langchain-chroma langchain-ollama beautifulsoup4 lxml

# If you hit: deepeval requires click<8.4.0 but click 8.4.1 is installed
# run the three commands below to fix and verify dependency conflicts.
!python -m pip install "click>=8.0.0,<8.4.0"
!python -m pip install --upgrade --force-reinstall "deepeval==4.0.4"
!python -m pip check

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
deepeval 4.0.4 requires click<8.4.0,>=8.0.0, but you have click 8.4.1 which is incompatible.
  Using cached click-8.3.3-py3-none-any.whl.metadata (2.6 kB)
Using cached click-8.3.3-py3-none-any.whl (110 kB)
  Attempting uninstall: click
    Found existing installation: click 8.4.1
    Uninstalling click-8.4.1:
      Successfully uninstalled click-8.4.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.16.4 requires click>=8.4.0, but you have click 8.3.3 which is incompatible.
  Using cached deepeval-4.0.4-py3-none-any.whl.metadata (27 kB)
  Using cached aiohttp-3.13.5-cp312-cp312-macosx_11_0_arm64.whl.metadata (8.1 kB)
  Using cached click-8.3.3-py3-none-any.whl.metadata (2.

In [2]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
from langchain_ollama import ChatOllama

/var/folders/90/zqrw_y6x2_74r6yr2q3qcrn00000gn/T/ipykernel_63687/2351560790.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
llm = ChatOllama(
    base_url="http://localhost:11434",
    model = "qwen2.5:latest",
    temperature=0.5,
    max_tokens = 250
)

In [4]:
# Load data from Web
loader = WebBaseLoader("https://www.descope.com/learn/post/mcp")
data = loader.load()

# Split text into documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(data)

# Add text to vector db
embedding = OllamaEmbeddings(model="nomic-embed-text:latest")
vectordb = Chroma.from_documents(documents=splits, embedding=embedding)

# Create a retriever
retriever = vectordb.as_retriever()

def format_docs(docs: List[Document]) -> str:
    return "\n\n".join([d.page_content for d in docs])


template = """Answer the question based only on the following context:

    {context}
    
    Give a summary not the full detail

    Question: {question}
    """
prompt = ChatPromptTemplate.from_template(template)


def retrieve_and_format(question):
    docs = retriever.invoke(question)
    return format_docs(docs)

chain = {"context": retrieve_and_format, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()


#### Output of the LLM Application

In [5]:
response = chain.invoke("What is MCP")

print(response)

MCP, or Model Context Protocol, standardizes how Language Models (LLMs) integrate with external systems. It works by streaming tool definitions—including their capabilities, data stores, and possible prompts—to LLMs from an MCP server. This protocol helps in defining and handling function schemas model-specifically and simplifies the interaction between LLMs and external tools through a client-server architecture.


### Testing RAG Application with DeepEval
<img src="./img/RAGTesting.png" width="800" height="400" style="display: block; margin: auto;">

In [6]:
import deepeval

deepeval.login("chf7LtTWtK1foTOAiK+vHFZ622I16kZtcpzfC+7FAVU=")

🎉🥳 Congratulations! You've successfully logged in! 🙌

In [7]:
!deepeval set-ollama --model deepseek-r1:8b

🙌 Congratulations! You're now using a local Ollama model `deepseek-r1:8b` for 
all evals that require an LLM.


In [8]:
test_data = [
    {
        "input": "What is MCP",
        "expected_output": "The Model Context Protocol (MCP) addresses this challenge by providing a standardized way for LLMs to connect with external data sources and tools—essentially a “universal remote” for AI apps. Released by Anthropic as an open-source protocol, MCP builds on existing function calling by eliminating the need for custom integration between LLMs and other apps."
    },
    {
        "input": "What is Relationship between function calling & Model Context Protocol",
        "expected_output": "The Model Context Protocol (MCP) builds on top of function calling, a well-established feature that allows large language models (LLMs) to invoke predetermined functions based on user requests. MCP simplifies and standardizes the development process by connecting AI applications to context while leveraging function calling to make API interactions more consistent across different applications and model vendors."
    },
    {
        "input": "What are the core components of MCP, just give the heading",
        "expected_output":""" 
                    - MCP Client
                    - MCP Servers
                    - Protocol Handshake
                    - Capability Discovery
                """
    }
]

### Creating Goldens

In [9]:
from deepeval.dataset import Golden, EvaluationDataset

goldens = []

for data in test_data:
    golden = Golden(
        input=data['input'],
        expected_output=data['expected_output']
    )
    
    goldens.append(golden)
    

dataset = EvaluationDataset(goldens=goldens)

In [10]:
dataset.push("test")

✅ Dataset successfully pushed to Confident AI! View at 
]8;id=14170432;https://app.confident-ai.com/project/cm8dzjvmw19bi7x4v8swskpku/datasets/cm8dznxvr19cg7x4vfai8djkh\https://app.confident-ai.com/project/cm8dzjvmw19bi7x4v8swskpku/datasets/cm8dznxvr19cg7x4vfai8djkh]8;;\

In [11]:
dataset

EvaluationDataset(test_cases=[], goldens=[Golden(input='What is MCP', actual_output=None, expected_output='The Model Context Protocol (MCP) addresses this challenge by providing a standardized way for LLMs to connect with external data sources and tools—essentially a “universal remote” for AI apps. Released by Anthropic as an open-source protocol, MCP builds on existing function calling by eliminating the need for custom integration between LLMs and other apps.', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None, multimodal=False, images_mapping=None), Golden(input='What is Relationship between function calling & Model Context Protocol', actual_output=None, expected_output='The Model Context Protocol (MCP) builds on top of function calling, a well-established feature that allows large language models (LLMs) to invoke predetermined functions based on user reque

In [12]:
dataset.pull(alias="test")

/Users/karthikkk/Downloads/aiqaDemoSource-2/.venv/lib/python3.12/site-packages/rich/live.py:260: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

In [13]:
dataset

EvaluationDataset(test_cases=[], goldens=[Golden(input='What is MCP', actual_output=None, expected_output='The Model Context Protocol (MCP) addresses this challenge by providing a standardized way for LLMs to connect with external data sources and tools—essentially a “universal remote” for AI apps. Released by Anthropic as an open-source protocol, MCP builds on existing function calling by eliminating the need for custom integration between LLMs and other apps.', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None, multimodal=False, images_mapping=None), Golden(input='What is Relationship between function calling & Model Context Protocol', actual_output=None, expected_output='The Model Context Protocol (MCP) builds on top of function calling, a well-established feature that allows large language models (LLMs) to invoke predetermined functions based on user reque

In [14]:
from langchain_classic.chains import RetrievalQA

# It is going to use the LLM and Vector database stored information (RAG)
qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)

In [15]:
response = qa_chain("What is MCP")

/var/folders/90/zqrw_y6x2_74r6yr2q3qcrn00000gn/T/ipykernel_63687/3706383010.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  response = qa_chain("What is MCP")


In [16]:
# Is the data which is stored in Vector DB
retrieved_document = retrieve_and_format("What is MCP")
print(retrieved_document)

## Why this matters

not mutually exclusive with MCP; the new protocol simply standardizes how this API feature works, adding context for the LLM. This is achieved by streaming tool definitions (including their capabilities, data stores, and possible prompts) to LLMs from the MCP server. Without MCP, when you use a function call directly with an LLM API, you need to:Define model-specific function schemas, which are JSON descriptions of the function, acceptable parameters, and what it returns.Implement handlers (the

What Is the Model Context Protocol (MCP) and How It WorksSkip to main contentArrow RightGet your complimentary copy of the Gartner Report: IAM Adapts to Secure and Enable AI Agents. Let's go >Log InUser CircleProductUse CasesDevelopersCustomersResourcesCompanyPricingSign upArrow RightBook a demoArrow Right# Contents

## Summary
Explains the Model Context Protocol (MCP), how it standardizes LLM integration with external systems, and its client-server architecture.

known as 

In [17]:
def query_with_context(question):
    retrieved_document = retrieve_and_format(question)
    response = qa_chain.run(question)
    return retrieved_document, response

In [18]:
actual, context = query_with_context("What is MCP")

actual, context

/var/folders/90/zqrw_y6x2_74r6yr2q3qcrn00000gn/T/ipykernel_63687/2296904857.py:3: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  response = qa_chain.run(question)


("## Why this matters\n\nnot mutually exclusive with MCP; the new protocol simply standardizes how this API feature works, adding context for the LLM. This is achieved by streaming tool definitions (including their capabilities, data stores, and possible prompts) to LLMs from the MCP server.\xa0Without MCP, when you use a function call directly with an LLM API, you need to:Define model-specific function schemas, which are JSON descriptions of the function, acceptable parameters, and what it returns.Implement handlers (the\n\nWhat Is the Model Context Protocol (MCP) and How It WorksSkip to main contentArrow RightGet your complimentary copy of the Gartner Report: IAM Adapts to Secure and Enable AI Agents. Let's go >Log InUser CircleProductUse CasesDevelopersCustomersResourcesCompanyPricingSign upArrow RightBook a demoArrow Right# Contents\n\n## Summary\nExplains the Model Context Protocol (MCP), how it standardizes LLM integration with external systems, and its client-server architecture

### Creating LLMTestCase with Goldens

In [19]:
from deepeval.dataset import EvaluationDataset
from deepeval.test_case import LLMTestCase
from typing import List


def convert_goldens_to_test_cases(dataset: EvaluationDataset) -> List[LLMTestCase]:
    test_cases = []
    for golden in dataset.goldens:
        context, rag_response = query_with_context(golden.input)
        test_case = LLMTestCase(
            input=golden.input,
            actual_output=rag_response,
            expected_output=golden.expected_output,
            retrieval_context=[context],
        )
        test_cases.append(test_case)
    return test_cases

data = convert_goldens_to_test_cases(dataset)

In [20]:
data

[LLMTestCase(input='What is MCP', actual_output='MCP stands for Model Context Protocol. It standardizes how language models, particularly Large Language Models (LLMs), integrate with external systems. Through this protocol, tool definitions—including their capabilities, data stores, and possible prompts—are streamed to LLMs from an MCP server. This helps in defining model-specific function schemas, acceptable parameters, and the expected outputs without needing to manually implement handlers for each interaction. Essentially, MCP simplifies and standardizes the process of integrating LLMs with various tools and systems by providing a structured way to pass context and information between them.', expected_output='The Model Context Protocol (MCP) addresses this challenge by providing a standardized way for LLMs to connect with external data sources and tools—essentially a “universal remote” for AI apps. Released by Anthropic as an open-source protocol, MCP builds on existing function cal

In [23]:
import deepeval.metrics
from deepeval.models import OllamaModel

deepeval.evaluate(
    data, 
    metrics= [
        deepeval.metrics.AnswerRelevancyMetric(model=OllamaModel(model="qwen2.5:latest")),
        deepeval.metrics.FaithfulnessMetric(model=OllamaModel(model="qwen2.5:latest"))
    ]
)

✨ You're running DeepEval's latest Answer Relevancy Metric! (using qwen2.5:latest (Ollama), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using qwen2.5:latest (Ollama), strict=False, 
async_mode=True)...

RetryError: RetryError[<Future at 0x11eb53950 state=finished raised TimeoutError>]